## Reusable Code from Day 1, Day 2 and Day 3 Tasks

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import ttest_rel
import time
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

adult = fetch_openml('adult', version=2, as_frame=True)
df = adult.frame
# Convert target labels to 0 and 1
df['class'] = df['class'].str.strip().map({'<=50K': 0, '>50K': 1})
# Replace string '?' placeholders with NaN if present
df.replace('?', np.nan, inplace=True)

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

X = df.drop(columns=['class'])
y = df['class']

# Stratified Hold-out Test Set (20%)
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Dev set (10% of total data = 12.5% of train_dev)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev, test_size=0.125, random_state=42, stratify=y_train_dev
)

MajorityModel = DummyClassifier(strategy='most_frequent')
MajorityModel.fit(X_train, y_train)

MajorityPred = MajorityModel.predict(X_test)
MajorityProb = MajorityModel.predict_proba(X_test)[:, 1]

# Single-Feature Rule Baseline: education-num >= 13
# Rationale: Higher education levels are generally associated with higher income,
# making education-num a reasonable simple feature for predicting income >50K.
# This rule is intentionally simple and provides a baseline for comparison with
# more advanced machine learning models.

RulePred = (X_test['education-num'] >= 13).astype(int)
RuleScore = X_test['education-num']

def EvaluateModel(YTrue, YPred, YScore):
    return {
        'Accuracy': accuracy_score(YTrue, YPred),
        'Precision': precision_score(YTrue, YPred, zero_division=0),
        'Recall': recall_score(YTrue, YPred, zero_division=0),
        'F1': f1_score(YTrue, YPred, zero_division=0),
        'ROC AUC': roc_auc_score(YTrue, YScore),
        'PR AUC': average_precision_score(YTrue, YScore)
    }

MajorityResults = EvaluateModel(
    y_test,
    MajorityPred,
    MajorityProb
)

RuleResults = EvaluateModel(
    y_test,
    RulePred,
    RuleScore
)

MetricsTable = pd.DataFrame(
    [MajorityResults, RuleResults],
    index=['Majority Class', 'Education Rule']
)

ErrorAnalysis = X_test.copy()
ErrorAnalysis['Actual'] = y_test
ErrorAnalysis['Predicted'] = RulePred

FalsePositives = ErrorAnalysis[
    (ErrorAnalysis['Actual'] == 0) &
    (ErrorAnalysis['Predicted'] == 1)
]

FalseNegatives = ErrorAnalysis[
    (ErrorAnalysis['Actual'] == 1) &
    (ErrorAnalysis['Predicted'] == 0)
]
num_cols = df.select_dtypes(include=[np.number]).columns
num_cols = num_cols.drop('class')  # Exclude target variable from features
cat_cols = df.select_dtypes(include=['object', 'category']).columns
NumericPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaler', StandardScaler())
])
CategoricalPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OneHotEncoder(handle_unknown='ignore'))
])

Preprocessor = ColumnTransformer([
    ('Numeric', NumericPipeline, num_cols),
    ('Categorical', CategoricalPipeline, cat_cols)
])

# Logistic Regression pipeline
logreg_pipeline = Pipeline([
    ("preprocessor", Preprocessor),
    ("model", LogisticRegression(
        random_state=42,
        solver="liblinear",
        max_iter=1000
    ))
])

# Decision Tree pipeline
dt_pipeline = Pipeline([
    ("preprocessor", Preprocessor),
    ("model", DecisionTreeClassifier(
        random_state=42
    ))
])

# Fit both models on the training set only
logreg_pipeline.fit(X_train, y_train)
dt_pipeline.fit(X_train, y_train)

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, auc
)

# Predictions and probabilities for both trained pipelines
logreg_pred = logreg_pipeline.predict(X_test)
logreg_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

dt_pred = dt_pipeline.predict(X_test)
dt_prob = dt_pipeline.predict_proba(X_test)[:, 1]

df['Age_Buckets'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 65, np.inf],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+'],
    include_lowest=True
)

# Part-time work is defined as fewer than 35 hours per week.
df['Is_PartTime'] = (df['hours-per-week'] < 35).astype('Int64')

# A flag distinguishes no capital gain from any capital gain.
df['Has_CapitalGain'] = (df['capital-gain'] > 0).astype('Int64')

# The log transformation reduces the influence of extremely large capital gains.
df['Log_CapitalGain'] = np.log1p(df['capital-gain'])

# This threshold identifies people with at least a bachelor's-level education.
df['Higher_Education'] = (df['education-num'] >= 13).astype('Int64')

# Working beyond a standard 40-hour week may signal higher earning potential.
df['Is_Overtime'] = (df['hours-per-week'] > 40).astype('Int64')

# Married categories represent a different household and life-stage pattern.
df['Is_Married'] = (
    df['marital-status'].fillna('').str.startswith('Married')
).astype('Int64')
FeatureDictionary = pd.DataFrame([
    {
        'Name': 'Age_Buckets',
        'Type': 'Categorical',
        'Creation_Rule': 'Bin age into six groups',
        'Predictive_Signal': 'Income may vary across career and retirement age groups'
    },
    {
        'Name': 'Is_PartTime',
        'Type': 'Binary',
        'Creation_Rule': 'hours-per-week < 35',
        'Predictive_Signal': 'Part-time workers may have lower income rates'
    },
    {
        'Name': 'Has_CapitalGain',
        'Type': 'Binary',
        'Creation_Rule': 'capital-gain > 0',
        'Predictive_Signal': 'Any capital gain may indicate higher earning capacity'
    },
    {
        'Name': 'Log_CapitalGain',
        'Type': 'Numeric',
        'Creation_Rule': 'log1p(capital-gain)',
        'Predictive_Signal': 'Capital-gain magnitude is retained while extreme values are compressed'
    },
    {
        'Name': 'Higher_Education',
        'Type': 'Binary',
        'Creation_Rule': 'education-num >= 13',
        'Predictive_Signal': 'Higher education is generally associated with higher income'
    },
    {
        'Name': 'Is_Overtime',
        'Type': 'Binary',
        'Creation_Rule': 'hours-per-week > 40',
        'Predictive_Signal': 'Overtime work may be associated with higher earnings'
    },
    {
        'Name': 'Is_Married',
        'Type': 'Binary',
        'Creation_Rule': 'marital-status starts with Married',
        'Predictive_Signal': 'Marital status may reflect household and life-stage differences'
    }
])
X = df.drop(columns=['class'])
y = df['class']

X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev,
    test_size=0.125,
    random_state=42,
    stratify=y_train_dev
)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

NumericPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaler', StandardScaler())
])
CategoricalPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OneHotEncoder(handle_unknown='ignore'))
])

Preprocessor = ColumnTransformer([
    ('Numeric', NumericPipeline, num_cols),
    ('Categorical', CategoricalPipeline, cat_cols)
])

# Reapply the row-wise feature rules whenever the pipeline receives new data.
def AddEngineeredFeatures(data):
    data = data.copy()
    data['Age_Buckets'] = pd.cut(
        data['age'],
        bins=[0, 25, 35, 45, 55, 65, np.inf],
        labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+'],
        include_lowest=True
    ).astype('object')
    data['Is_PartTime'] = (data['hours-per-week'] < 35).astype('float64')
    data['Has_CapitalGain'] = (data['capital-gain'] > 0).astype('float64')
    data['Log_CapitalGain'] = np.log1p(data['capital-gain'])
    data['Higher_Education'] = (data['education-num'] >= 13).astype('float64')
    data['Is_Overtime'] = (data['hours-per-week'] > 40).astype('float64')
    data['Is_Married'] = data['marital-status'].fillna('').str.startswith('Married').astype('float64')
    return data

FeatureEngineeringTransformer = FunctionTransformer(
    AddEngineeredFeatures,
    validate=False
)

# The transformer runs before preprocessing and the model is fitted on training data only.
logreg_pipeline = Pipeline([
    ('feature_engineering', FeatureEngineeringTransformer),
    ('preprocessor', Preprocessor),
    ('model', LogisticRegression(
        random_state=42,
        solver='liblinear',
        max_iter=1000
    ))
])

dt_pipeline = Pipeline([
    ('feature_engineering', FeatureEngineeringTransformer),
    ('preprocessor', Preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

logreg_pipeline.fit(X_train, y_train)
dt_pipeline.fit(X_train, y_train)

EngineeredFeatureNames = [
    'Age_Buckets',
    'Is_PartTime',
    'Has_CapitalGain',
    'Log_CapitalGain',
    'Higher_Education',
    'Is_Overtime',
    'Is_Married'
]

# Calculate univariate signal on training-development rows only.
FeatureRows = X_train_dev[EngineeredFeatureNames].copy()
UnivariateScores = []

for FeatureName in EngineeredFeatureNames:
    FeatureValues = FeatureRows[[FeatureName]].copy()
    if FeatureValues[FeatureName].dtype.name in ['object', 'category']:
        FeatureValues[FeatureName] = (
            FeatureValues[FeatureName]
            .fillna('Missing')
            .astype('category')
            .cat.codes
        )
    else:
        FeatureValues[FeatureName] = FeatureValues[FeatureName].fillna(
            FeatureValues[FeatureName].median()
        )

    MutualInformation = mutual_info_classif(
        FeatureValues,
        y_train_dev,
        discrete_features=(FeatureValues[FeatureName].nunique() <= 10),
        random_state=42
    )[0]
    GroupedTargetRateRange = (
        pd.DataFrame({
            'Feature': FeatureRows[FeatureName],
            'Target': y_train_dev
        })
        .groupby('Feature', observed=True)['Target']
        .mean()
        .pipe(lambda rates: rates.max() - rates.min())
    )
    UnivariateScores.append({
        'Name': FeatureName,
        'Mutual_Information': MutualInformation,
        'Grouped_Target_Rate_Range': GroupedTargetRateRange
    })

UnivariateScores = pd.DataFrame(UnivariateScores)
FeatureDictionary = FeatureDictionary.merge(UnivariateScores, on='Name')
FeatureDictionary.sort_values('Mutual_Information', ascending=False)

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Use the engineered dataset and keep the final test set separate from cross-validation.
CVNumericPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaler', StandardScaler())
])

CVCategoricalPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

CVPreprocessor = ColumnTransformer([
    ('Numeric', CVNumericPipeline, num_cols),
    ('Categorical', CVCategoricalPipeline, cat_cols)
])

Models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        solver='liblinear',
        max_iter=1000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Scoring = {
    'Accuracy': 'accuracy',
    'ROC_AUC': 'roc_auc',
    'F1': 'f1'
}

CVResults = []
FoldScores = []

for ModelName, Model in Models.items():
    ModelPipeline = Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', clone(Model))
    ])
    Scores = cross_validate(
        ModelPipeline,
        X_train_dev,
        y_train_dev,
        cv=CV,
        scoring=Scoring,
        n_jobs=-1,
        return_train_score=False
    )

    for FoldNumber in range(5):
        FoldScores.append({
            'Model': ModelName,
            'Fold': FoldNumber + 1,
            'Accuracy': Scores['test_Accuracy'][FoldNumber],
            'ROC AUC': Scores['test_ROC_AUC'][FoldNumber],
            'F1': Scores['test_F1'][FoldNumber]
        })

    CVResults.append({
        'Model': ModelName,
        'Accuracy Mean': Scores['test_Accuracy'].mean(),
        'Accuracy Std': Scores['test_Accuracy'].std(),
        'ROC AUC Mean': Scores['test_ROC_AUC'].mean(),
        'ROC AUC Std': Scores['test_ROC_AUC'].std(),
        'F1 Mean': Scores['test_F1'].mean(),
        'F1 Std': Scores['test_F1'].std()
    })

CVSummary = pd.DataFrame(CVResults).set_index('Model')
FoldScores = pd.DataFrame(FoldScores)

print('Cross-validated performance (mean +/- standard deviation):')
CVSummary.round(4)

from scipy.stats import ttest_rel

# F1 is the primary metric, so compare the two models with the highest mean F1.
F1Scores = FoldScores.pivot(index='Fold', columns='Model', values='F1')
TopModels = ['Gradient Boosting', 'Logistic Regression']
PairedTest = ttest_rel(F1Scores[TopModels[0]], F1Scores[TopModels[1]])
MeanDifference = (
    F1Scores[TopModels[0]] - F1Scores[TopModels[1]]
).mean()

StatisticalComparison = pd.DataFrame({
    'Metric': ['F1'],
    'Top Model': [TopModels[0]],
    'Second Model': [TopModels[1]],
    'Top Mean': [F1Scores[TopModels[0]].mean()],
    'Second Mean': [F1Scores[TopModels[1]].mean()],
    'Mean Difference': [MeanDifference],
    'Paired t-statistic': [PairedTest.statistic],
    'p-value': [PairedTest.pvalue]
})

print('Paired statistical comparison using F1:')
print(StatisticalComparison.round(6).to_string(index=False))
print(
    f'Practical interpretation: Gradient Boosting improves mean F1 by '
    f'{MeanDifference:.4f} ({MeanDifference * 100:.2f} percentage points). '
    f'This is statistically significant at alpha=0.05: '
    f'{PairedTest.pvalue < 0.05}, but the practical gain is modest.'
)

# Fit the top two models on all training-development rows for feature analysis.
FinalGradientBoostingPipeline = Pipeline([
    ('Preprocessor', clone(CVPreprocessor)),
    ('Model', GradientBoostingClassifier(random_state=42))
])
FinalLogisticPipeline = Pipeline([
    ('Preprocessor', clone(CVPreprocessor)),
    ('Model', LogisticRegression(
        random_state=42,
        solver='liblinear',
        max_iter=1000
    ))
])

FinalGradientBoostingPipeline.fit(X_train_dev, y_train_dev)
FinalLogisticPipeline.fit(X_train_dev, y_train_dev)

TransformedFeatureNames = FinalGradientBoostingPipeline.named_steps[
    'Preprocessor'
].get_feature_names_out()
TreeImportances = FinalGradientBoostingPipeline.named_steps['Model'].feature_importances_
LogisticCoefficients = FinalLogisticPipeline.named_steps['Model'].coef_[0]

ImportanceTable = pd.DataFrame({
    'Transformed_Feature': TransformedFeatureNames,
    'Tree_Importance': TreeImportances,
    'Logistic_Coefficient': LogisticCoefficients,
    'Absolute_Logistic_Coefficient': np.abs(LogisticCoefficients)
})

EngineeredImportanceTable = ImportanceTable[
    ImportanceTable['Transformed_Feature'].str.contains(
        'Age_Buckets|Is_PartTime|Has_CapitalGain|Log_CapitalGain|'
        'Higher_Education|Is_Overtime|Is_Married',
        regex=True
    )
].copy()
EngineeredImportanceTable['Engineered_Feature'] = (
    EngineeredImportanceTable['Transformed_Feature']
    .str.replace(r'^Categorical__Age_Buckets_.*$', 'Age_Buckets', regex=True)
    .str.replace(r'^Numeric__', '', regex=True)
)

FeatureImportanceSummary = EngineeredImportanceTable.groupby(
    'Engineered_Feature', as_index=False
).agg(
    Tree_Importance=('Tree_Importance', 'sum'),
    Maximum_Absolute_Logistic_Coefficient=(
        'Absolute_Logistic_Coefficient', 'max'
    )
).sort_values('Tree_Importance', ascending=False)

print('\nEngineered feature importance summary:')
FeatureImportanceSummary.round(6)
# Detailed coefficients and tree importances for every engineered transformed column.
EngineeredImportanceTable[
    [
        'Engineered_Feature',
        'Transformed_Feature',
        'Tree_Importance',
        'Logistic_Coefficient'
    ]
].sort_values('Tree_Importance', ascending=False).round(6)

# Compare the full engineered representation with mutual-information selection.
SelectionModels = {
    'Full features': 'passthrough',
    'SelectKBest (k=50)': SelectKBest(
        score_func=mutual_info_classif,
        k=50
    )
}

SelectionResults = []
SelectionFoldScores = []

for SelectionName, SelectionStep in SelectionModels.items():
    SelectionPipeline = Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Selection', clone(SelectionStep) if SelectionStep != 'passthrough' else 'passthrough'),
        ('Model', GradientBoostingClassifier(random_state=42))
    ])

    StartTime = time.perf_counter()
    SelectionScores = cross_validate(
        SelectionPipeline,
        X_train_dev,
        y_train_dev,
        cv=CV,
        scoring={'F1': 'f1', 'ROC_AUC': 'roc_auc', 'Accuracy': 'accuracy'},
        n_jobs=-1,
        return_train_score=False
    )
    ElapsedTime = time.perf_counter() - StartTime

    for FoldNumber in range(5):
        SelectionFoldScores.append({
            'Selection': SelectionName,
            'Fold': FoldNumber + 1,
            'F1': SelectionScores['test_F1'][FoldNumber],
            'ROC AUC': SelectionScores['test_ROC_AUC'][FoldNumber],
            'Accuracy': SelectionScores['test_Accuracy'][FoldNumber]
        })

    SelectionResults.append({
        'Selection': SelectionName,
        'F1 Mean': SelectionScores['test_F1'].mean(),
        'F1 Std': SelectionScores['test_F1'].std(),
        'ROC AUC Mean': SelectionScores['test_ROC_AUC'].mean(),
        'ROC AUC Std': SelectionScores['test_ROC_AUC'].std(),
        'Accuracy Mean': SelectionScores['test_Accuracy'].mean(),
        'Accuracy Std': SelectionScores['test_Accuracy'].std(),
        'Mean Fit Time Seconds': SelectionScores['fit_time'].mean(),
        'Total CV Time Seconds': ElapsedTime
    })


SelectionSummary = pd.DataFrame(SelectionResults).set_index('Selection')
SelectionFoldScores = pd.DataFrame(SelectionFoldScores)

print('Feature-selection comparison:')
SelectionSummary.round(4)

# Task 1

In [ ]:
from pathlib import Path
import json
import platform
import joblib
import sklearn

RANDOM_STATE = 42
ARTIFACT_PATH = Path('reproducible_adult_pipelines.joblib')

# Rebuild the candidate pipelines so preprocessing is fitted inside each pipeline.
ReproduciblePipelines = {
    'Logistic Regression': Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', LogisticRegression(
            solver='liblinear',
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

for ModelPipeline in ReproduciblePipelines.values():
    ModelPipeline.fit(X_train_dev, y_train_dev)

ReproducibleMetrics = {}
for ModelName, ModelPipeline in ReproduciblePipelines.items():
    TestPredictions = ModelPipeline.predict(X_test)
    TestProbabilities = ModelPipeline.predict_proba(X_test)[:, 1]
    ReproducibleMetrics[ModelName] = {
        'Accuracy': accuracy_score(y_test, TestPredictions),
        'F1': f1_score(y_test, TestPredictions),
        'ROC AUC': roc_auc_score(y_test, TestProbabilities)
    }

VersionMetadata = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'random_state': RANDOM_STATE,
    'dataset': 'OpenML Adult, version 2',
    'training_rows': len(X_train_dev),
    'test_rows': len(X_test)
}

PipelineBundle = {
    'pipelines': ReproduciblePipelines,
    'metrics': ReproducibleMetrics,
    'metadata': VersionMetadata
}
joblib.dump(PipelineBundle, ARTIFACT_PATH)

ReloadedBundle = joblib.load(ARTIFACT_PATH)
ReloadCheck = {
    ModelName: len(ModelPipeline.predict(X_test))
    for ModelName, ModelPipeline in ReloadedBundle['pipelines'].items()
}
assert all(RowCount == len(X_test) for RowCount in ReloadCheck.values())

print(f'Saved reproducible pipelines to: {ARTIFACT_PATH.resolve()}')
print(json.dumps(VersionMetadata, indent=2))
pd.DataFrame(ReproducibleMetrics).T.round(4)


## Reproducibility README

This notebook uses fixed random seeds and sklearn Pipelines to make model training reproducible. The same stratified train/test split (`random_state=42`) and 5-fold `StratifiedKFold` cross-validation are used throughout the experiments. Preprocessing, feature transformation, and model fitting are contained inside each pipeline to prevent data leakage during cross-validation.

To reproduce the results:

1. Run the notebook from the beginning so the Adult dataset is loaded and the train/development/test splits are recreated.
2. Run the feature-engineering and preprocessing cells.
3. Run the cross-validation and hyperparameter-search cells.
4. Run the final reproducible pipeline cell to train the selected models and save them.
5. The trained pipelines are saved as `reproducible_adult_pipelines.joblib`.
6. Loading this file with `joblib.load()` restores the fitted preprocessing and model pipeline.

The experiment records the Python, NumPy, pandas, and scikit-learn versions together with the random seed and dataset version so that the environment used for training is documented.


# Task 2

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Search each estimator behind the same preprocessing pipeline.
SearchPipelines = {
    'Logistic Regression': Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', LogisticRegression(
            solver='liblinear',
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
    'Random Forest': Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

SearchSpaces = {
    'Logistic Regression': {
        'Model__penalty': ['l1', 'l2'],
        'Model__C': [0.01, 0.03, 0.1, 0.3, 1, 3]
    },
    'Random Forest': {
        'Model__n_estimators': [200, 400, 600],
        'Model__max_depth': [None, 10, 20, 30],
        'Model__min_samples_leaf': [1, 2, 5, 10],
        'Model__max_features': ['sqrt', 'log2', 0.5, 0.8]
    },
    'Gradient Boosting': {
        'Model__n_estimators': [100, 150, 200, 300, 400],
        'Model__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
        'Model__max_depth': [1, 2, 3, 4, 5],
        'Model__min_samples_leaf': [1, 2, 5, 10]
    }
}

SearchBudgets = {
    'Logistic Regression': 6,
    'Random Forest': 15,
    'Gradient Boosting': 15
}

SearchResults = {}
SearchSummaryRows = []
for ModelName, SearchPipeline in SearchPipelines.items():
    Search = RandomizedSearchCV(
    estimator=SearchPipeline,
    param_distributions=SearchSpaces[ModelName],
    n_iter=SearchBudgets[ModelName],
    scoring='f1',
    cv=CV,
    n_jobs=2,
    random_state=RANDOM_STATE,
    refit=True,
    return_train_score=False
)
    Search.fit(X_train_dev, y_train_dev)
    SearchResults[ModelName] = Search
    SearchSummaryRows.append({
        'Model': ModelName,
        'Best F1 CV': Search.best_score_,
        'Best Parameters': Search.best_params_,
        'Candidates Evaluated': len(Search.cv_results_['params'])
    })

SearchSummary = pd.DataFrame(SearchSummaryRows).set_index('Model')
print('Randomized search results (primary metric: F1):')
SearchSummary[['Best F1 CV', 'Candidates Evaluated']].round(4)
print('\nBest parameters:')
for ModelName, Search in SearchResults.items():
    print(f'{ModelName}: {Search.best_params_}')


# Task 3

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LearningCurvePipeline = Pipeline([
    ('Preprocessor', clone(CVPreprocessor)),
    ('Model', LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

TrainSizes, TrainScores, ValidationScores = learning_curve(
    LearningCurvePipeline,
    X_train_dev,
    y_train_dev,
    cv=CV,
    scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 6),
    n_jobs=2
)

TrainMean = TrainScores.mean(axis=1)
TrainStd = TrainScores.std(axis=1)
ValidationMean = ValidationScores.mean(axis=1)
ValidationStd = ValidationScores.std(axis=1)

LearningCurveResults = pd.DataFrame({
    'Training Size': TrainSizes,
    'Train F1': TrainMean,
    'Validation F1': ValidationMean
})

print('Learning Curve Results:')
print(LearningCurveResults.round(4))

plt.figure(figsize=(9, 6))
plt.plot(TrainSizes, TrainMean, marker='o', label='Training F1')
plt.plot(TrainSizes, ValidationMean, marker='o', label='Validation F1')
plt.fill_between(
    TrainSizes,
    TrainMean - TrainStd,
    TrainMean + TrainStd,
    alpha=0.15
)
plt.fill_between(
    TrainSizes,
    ValidationMean - ValidationStd,
    ValidationMean + ValidationStd,
    alpha=0.15
)
plt.xlabel('Number of Training Examples')
plt.ylabel('F1 Score')
plt.title('Logistic Regression Learning Curve')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
CValues = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
CResults = []

for CValue in CValues:
    CModelPipeline = Pipeline([
        ('Preprocessor', clone(CVPreprocessor)),
        ('Model', LogisticRegression(
            solver='liblinear',
            C=CValue,
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ])
    CValidation = cross_validate(
        CModelPipeline,
        X_train_dev,
        y_train_dev,
        cv=CV,
        scoring='f1',
        n_jobs=2,
        return_train_score=True
    )
    CResults.append({
        'C': CValue,
        'Train F1': CValidation['train_score'].mean(),
        'Validation F1': CValidation['test_score'].mean()
    })

CResultsData = pd.DataFrame(CResults)

print('Effect of C on Logistic Regression:')
print(CResultsData.round(4))

plt.figure(figsize=(9, 6))
plt.semilogx(CResultsData['C'], CResultsData['Train F1'], marker='o', label='Training F1')
plt.semilogx(CResultsData['C'], CResultsData['Validation F1'], marker='o', label='Validation F1')
plt.xlabel('C (Log Scale)')
plt.ylabel('F1 Score')
plt.title('Effect of Regularization Strength on Logistic Regression')
plt.legend()
plt.grid(True)
plt.show()

### Task 3: Overfitting / Underfitting Analysis

The Logistic Regression learning curve shows the relationship between training size and F1 score. The training score is higher than the validation score, indicating some degree of variance. As the training set increases, the validation score improves and the gap between training and validation performance decreases. This suggests that additional training data could help improve generalization.

The regularization analysis shows how the Logistic Regression parameter C affects performance. Smaller C values apply stronger regularization, while larger C values allow a more complex model. The best region should be selected based on validation F1 rather than training F1, since F1 is the primary metric for this project.

Based on these results, the main fix is to select the C value that provides the strongest validation F1 while avoiding a large train-validation gap. If the learning curve continues improving with more training examples, adding more data would also be beneficial. If the train-validation gap remains large, stronger regularization through a smaller C should be tested.

# Task 4

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BestLogisticModel = SearchResults['Logistic Regression'].best_estimator_

BestLogisticModel.fit(X_train, y_train)

DevProbabilities = BestLogisticModel.predict_proba(X_dev)[:, 1]
TestProbabilities = BestLogisticModel.predict_proba(X_test)[:, 1]

DevBrierScore = brier_score_loss(y_dev, DevProbabilities)
TestBrierScore = brier_score_loss(y_test, TestProbabilities)

print(f'Development Brier Score: {DevBrierScore:.4f}')
print(f'Test Brier Score: {TestBrierScore:.4f}')

DevFractionOfPositives, DevMeanPredictedValue = calibration_curve(
    y_dev,
    DevProbabilities,
    n_bins=10,
    strategy='quantile'
)

plt.figure(figsize=(8, 6))
plt.plot(
    DevMeanPredictedValue,
    DevFractionOfPositives,
    marker='o',
    label='Logistic Regression'
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Perfect Calibration'
)
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Probability Calibration Plot')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

CalibratedModel = CalibratedClassifierCV(
    BestLogisticModel,
    method='sigmoid',
    cv=CV
)

CalibratedModel.fit(X_train, y_train)

CalibratedDevProbabilities = CalibratedModel.predict_proba(X_dev)[:, 1]
CalibratedTestProbabilities = CalibratedModel.predict_proba(X_test)[:, 1]

CalibratedDevBrierScore = brier_score_loss(
    y_dev,
    CalibratedDevProbabilities
)

CalibratedTestBrierScore = brier_score_loss(
    y_test,
    CalibratedTestProbabilities
)

print(f'Original Development Brier Score: {DevBrierScore:.4f}')
print(f'Calibrated Development Brier Score: {CalibratedDevBrierScore:.4f}')
print(f'Original Test Brier Score: {TestBrierScore:.4f}')
print(f'Calibrated Test Brier Score: {CalibratedTestBrierScore:.4f}')
FinalProbabilityModel = CalibratedModel
DevProbabilities = CalibratedDevProbabilities
TestProbabilities = CalibratedTestProbabilities

In [ ]:
ThresholdValues = np.arange(0.10, 0.91, 0.05)
ThresholdResults = []

for Threshold in ThresholdValues:
    DevPredictions = (DevProbabilities >= Threshold).astype(int)
    ThresholdResults.append({
        'Threshold': Threshold,
        'Accuracy': accuracy_score(y_dev, DevPredictions),
        'Precision': precision_score(y_dev, DevPredictions, zero_division=0),
        'Recall': recall_score(y_dev, DevPredictions, zero_division=0),
        'F1': f1_score(y_dev, DevPredictions, zero_division=0)
    })

ThresholdResultsData = pd.DataFrame(ThresholdResults)

print('Development Set Threshold Performance:')
print(ThresholdResultsData.round(4))

BestThresholdRow = ThresholdResultsData.loc[
    ThresholdResultsData['F1'].idxmax()
]

BestThreshold = BestThresholdRow['Threshold']

print(f'\nSelected Threshold: {BestThreshold:.2f}')
print(f"Development F1: {BestThresholdRow['F1']:.4f}")
print(f"Development Precision: {BestThresholdRow['Precision']:.4f}")
print(f"Development Recall: {BestThresholdRow['Recall']:.4f}")

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(
    ThresholdResultsData['Threshold'],
    ThresholdResultsData['Precision'],
    marker='o',
    label='Precision'
)
plt.plot(
    ThresholdResultsData['Threshold'],
    ThresholdResultsData['Recall'],
    marker='o',
    label='Recall'
)
plt.plot(
    ThresholdResultsData['Threshold'],
    ThresholdResultsData['F1'],
    marker='o',
    label='F1'
)
plt.xlabel('Classification Threshold')
plt.ylabel('Score')
plt.title('Threshold Selection on Development Set')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
DefaultThreshold = 0.50

DefaultTestPredictions = (
    TestProbabilities >= DefaultThreshold
).astype(int)

OptimizedTestPredictions = (
    TestProbabilities >= BestThreshold
).astype(int)

DefaultTestMatrix = confusion_matrix(
    y_test,
    DefaultTestPredictions
)

OptimizedTestMatrix = confusion_matrix(
    y_test,
    OptimizedTestPredictions
)

print('Confusion Matrix - Default Threshold 0.50:')
print(DefaultTestMatrix)

print(f'\nConfusion Matrix - Optimized Threshold {BestThreshold:.2f}:')
print(OptimizedTestMatrix)

In [ ]:
DefaultTN, DefaultFP, DefaultFN, DefaultTP = DefaultTestMatrix.ravel()
OptimizedTN, OptimizedFP, OptimizedFN, OptimizedTP = OptimizedTestMatrix.ravel()

BusinessComparison = pd.DataFrame({
    'KPI': [
        'True Positives',
        'False Positives',
        'False Negatives',
        'True Negatives',
        'Accuracy',
        'Precision',
        'Recall',
        'F1'
    ],
    'Threshold 0.50': [
        DefaultTP,
        DefaultFP,
        DefaultFN,
        DefaultTN,
        accuracy_score(y_test, DefaultTestPredictions),
        precision_score(y_test, DefaultTestPredictions, zero_division=0),
        recall_score(y_test, DefaultTestPredictions, zero_division=0),
        f1_score(y_test, DefaultTestPredictions, zero_division=0)
    ],
    'Optimized Threshold': [
        OptimizedTP,
        OptimizedFP,
        OptimizedFN,
        OptimizedTN,
        accuracy_score(y_test, OptimizedTestPredictions),
        precision_score(y_test, OptimizedTestPredictions, zero_division=0),
        recall_score(y_test, OptimizedTestPredictions, zero_division=0),
        f1_score(y_test, OptimizedTestPredictions, zero_division=0)
    ]
})

print('Final Test Business KPI Comparison:')
print(BusinessComparison.round(4))

### Task 4: Probability Calibration & Threshold Selection

Probability calibration was evaluated using a calibration plot and the Brier score. The Brier score measures the quality of predicted probabilities, with lower values indicating better calibration. The calibration plot compares predicted probabilities with the observed fraction of positive cases.

The classification threshold was selected using the development set rather than the test set to avoid test-set leakage. Multiple thresholds from 0.10 to 0.90 were evaluated using F1 as the primary business metric. The threshold with the highest development-set F1 was selected as the optimized threshold.

The optimized threshold was then applied to the untouched test set and compared with the default threshold of 0.50. Lower thresholds generally increase recall and identify more potential high-income individuals, while higher thresholds generally increase precision and reduce unnecessary outreach. In this targeted marketing scenario, false positives represent potentially wasted marketing outreach, while false negatives represent missed high-income prospects.

The final threshold was therefore selected to maximize F1 while maintaining a reasonable balance between precision and recall. The test-set results were used only for final evaluation after the threshold had been selected.

# Task 5

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

FinalModel = SearchResults['Logistic Regression'].best_estimator_

FinalPipeline = FinalModel

FinalPipeline.fit(X_train, y_train)

FinalTestProbabilities = FinalPipeline.predict_proba(X_test)[:, 1]

FinalDefaultPredictions = (
    FinalTestProbabilities >= 0.50
).astype(int)

FinalOptimizedPredictions = (
    FinalTestProbabilities >= BestThreshold
).astype(int)
FinalPipeline = CalibratedModel
FinalPipeline.fit(X_train, y_train)
FinalTestProbabilities = FinalPipeline.predict_proba(X_test)[:, 1]
FinalOptimizedPredictions = (
    FinalTestProbabilities >= BestThreshold
).astype(int)

In [ ]:
FinalMetrics = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC AUC',
        'Average Precision',
        'Brier Score'
    ],
    'Default Threshold 0.50': [
        accuracy_score(y_test, FinalDefaultPredictions),
        precision_score(y_test, FinalDefaultPredictions, zero_division=0),
        recall_score(y_test, FinalDefaultPredictions, zero_division=0),
        f1_score(y_test, FinalDefaultPredictions, zero_division=0),
        roc_auc_score(y_test, FinalTestProbabilities),
        average_precision_score(y_test, FinalTestProbabilities),
        brier_score_loss(y_test, FinalTestProbabilities)
    ],
    f'Optimized Threshold {BestThreshold:.2f}': [
        accuracy_score(y_test, FinalOptimizedPredictions),
        precision_score(y_test, FinalOptimizedPredictions, zero_division=0),
        recall_score(y_test, FinalOptimizedPredictions, zero_division=0),
        f1_score(y_test, FinalOptimizedPredictions, zero_division=0),
        roc_auc_score(y_test, FinalTestProbabilities),
        average_precision_score(y_test, FinalTestProbabilities),
        brier_score_loss(y_test, FinalTestProbabilities)
    ]
})

print('Final Test Metrics:')
print(FinalMetrics.round(4))

In [ ]:
FinalConfusionMatrix = confusion_matrix(
    y_test,
    FinalOptimizedPredictions
)

FinalTN, FinalFP, FinalFN, FinalTP = FinalConfusionMatrix.ravel()

print(f'Final Confusion Matrix at Threshold {BestThreshold:.2f}:')
print(FinalConfusionMatrix)

print('\nTrue Negatives:', FinalTN)
print('False Positives:', FinalFP)
print('False Negatives:', FinalFN)
print('True Positives:', FinalTP)

In [ ]:
plt.figure(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test,
    FinalTestProbabilities
)

plt.title('Final Model ROC Curve')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

PrecisionRecallDisplay.from_predictions(
    y_test,
    FinalTestProbabilities
)

plt.title('Final Model Precision-Recall Curve')
plt.grid(True)
plt.show()

In [ ]:
FinalArtifactPath = Path('final_adult_income_pipeline.joblib')

FinalArtifact = {
    'pipeline': FinalPipeline,
    'threshold': float(BestThreshold),
    'primary_metric': 'F1',
    'model': 'Logistic Regression',
    'best_parameters': SearchResults['Logistic Regression'].best_params_,
    'test_metrics': {
        'Accuracy': accuracy_score(y_test, FinalOptimizedPredictions),
        'Precision': precision_score(y_test, FinalOptimizedPredictions, zero_division=0),
        'Recall': recall_score(y_test, FinalOptimizedPredictions, zero_division=0),
        'F1': f1_score(y_test, FinalOptimizedPredictions, zero_division=0),
        'ROC AUC': roc_auc_score(y_test, FinalTestProbabilities),
        'Average Precision': average_precision_score(y_test, FinalTestProbabilities),
        'Brier Score': brier_score_loss(y_test, FinalTestProbabilities)
    },
    'random_state': RANDOM_STATE
}

joblib.dump(FinalArtifact, FinalArtifactPath)

print(f'Final artifact saved to: {FinalArtifactPath.resolve()}')

In [ ]:
ReloadedArtifact = joblib.load(FinalArtifactPath)

ReloadedPipeline = ReloadedArtifact['pipeline']
ReloadedThreshold = ReloadedArtifact['threshold']

ReloadedProbabilities = ReloadedPipeline.predict_proba(X_test)[:, 1]

ReloadedPredictions = (
    ReloadedProbabilities >= ReloadedThreshold
).astype(int)

ReloadedF1 = f1_score(
    y_test,
    ReloadedPredictions
)

print('Reloaded artifact verification:')
print('Saved threshold:', ReloadedThreshold)
print('Reloaded test F1:', round(ReloadedF1, 4))

assert len(ReloadedPredictions) == len(y_test)

## How to Use the Saved Pipeline for Inference

The saved `final_adult_income_pipeline.joblib` file contains the complete preprocessing pipeline, trained Logistic Regression model, and optimized classification threshold.

```python
import joblib
import pandas as pd

Artifact = joblib.load('final_adult_income_pipeline.joblib')

Pipeline = Artifact['pipeline']
Threshold = Artifact['threshold']

NewData = pd.DataFrame({
    # Add the same feature columns used during training
})

Probabilities = Pipeline.predict_proba(NewData)[:, 1]

Predictions = (Probabilities >= Threshold).astype(int)

Results = NewData.copy()
Results['ProbabilityAbove50K'] = Probabilities
Results['PredictedClass'] = Predictions

print(Results)


---

# 5H. Final summary cell

After running everything, add this Markdown cell.

**Don't put the actual numbers in manually until you have your output.**

```markdown
## Task 5: Final Evaluation & Artifact Summary

### Selected Model

The final model was Logistic Regression, selected based on cross-validation F1 performance during hyperparameter tuning. The preprocessing and estimator were kept together in a single sklearn Pipeline to ensure that the same transformations are applied consistently during training and inference.

### Hyperparameter Tuning

RandomizedSearchCV was used with 5-fold StratifiedKFold cross-validation and F1 as the primary optimization metric. The selected hyperparameters were:

- C: [INSERT BEST C]
- Penalty: [INSERT BEST PENALTY]
- Solver: liblinear
- Maximum iterations: 1000

The final classification threshold was selected using the development set to maximize F1. The test set was not used for threshold selection.

### Final Test Performance

At the optimized threshold of [INSERT THRESHOLD], the final model achieved:

- Accuracy: [INSERT VALUE]
- Precision: [INSERT VALUE]
- Recall: [INSERT VALUE]
- F1: [INSERT VALUE]
- ROC AUC: [INSERT VALUE]
- Average Precision: [INSERT VALUE]
- Brier Score: [INSERT VALUE]

The confusion matrix contained [INSERT TP] true positives, [INSERT FP] false positives, [INSERT FN] false negatives, and [INSERT TN] true negatives.

### Production Behavior

In production, the pipeline accepts new observations, applies the saved preprocessing transformations, generates a probability of earning more than $50K, and converts that probability into a classification using the optimized threshold. The threshold can be adjusted later if the business changes the relative cost of false positives and false negatives. Model performance and probability calibration should be monitored over time because the underlying population and data distribution may change.

The complete trained pipeline and threshold were saved using `joblib` in `final_adult_income_pipeline.joblib`.